# Sesión 2 (complemento). RSA y LFSR en Python

En este notebook vamos a **ejecutar en Python** dos piezas fundamentales de la criptografía:

1. **RSA**: el algoritmo de clave pública visto en la sesión 2, con el mismo ejemplo numérico didáctico ($p=5$, $q=11$).
2. **LFSR** (*Linear Feedback Shift Register*, registro de desplazamiento con realimentación lineal): un generador de secuencias pseudoaleatorias muy sencillo, histórico en los cifrados de flujo, y muy útil para entender **por qué "parecer aleatorio" no es lo mismo que ser criptográficamente seguro**.

El objetivo es puramente didáctico: usar números pequeños y código simple para poder seguir cada paso a mano si se quiere.

## Objetivos
- Generar un par de claves RSA de juguete y cifrar/descifrar un mensaje.
- Firmar y verificar un mensaje con RSA de forma simplificada.
- Implementar un LFSR desde cero y generar un flujo de bits.
- Observar el periodo del LFSR y entender por qué un LFSR no es seguro por sí solo.

---
## Parte 1. RSA paso a paso

Recordamos el procedimiento visto en clase:

1. Elegir dos primos $p$ y $q$.
2. Calcular $n = p \times q$.
3. Calcular $\varphi(n) = (p-1)(q-1)$.
4. Elegir un exponente público $e$ tal que $\gcd(e, \varphi(n)) = 1$.
5. Calcular $d$, el inverso modular de $e$ módulo $\varphi(n)$.
6. Clave pública: $(e, n)$. Clave privada: $(d, n)$.

Vamos a implementarlo con Python usando los mismos números del temario: $p = 5$, $q = 11$.

In [1]:
from math import gcd

def generar_claves_rsa(p, q, e=None):
    """Genera un par de claves RSA de juguete a partir de dos primos pequeños p y q.

    Devuelve un diccionario con n, phi, e, d, clave publica y clave privada.
    Uso EXCLUSIVAMENTE didáctico: en RSA real, p y q tienen cientos de dígitos.
    """
    n = p * q
    phi = (p - 1) * (q - 1)

    if e is None:
        # Buscamos el primer e > 1 que sea primo relativo con phi
        e = 2
        while gcd(e, phi) != 1:
            e += 1

    if gcd(e, phi) != 1:
        raise ValueError(f"e={e} no es válido: debe cumplir gcd(e, phi(n)) = 1")

    d = pow(e, -1, phi)  # inverso modular de e mod phi (Python 3.8+)

    return {
        "p": p, "q": q, "n": n, "phi": phi,
        "e": e, "d": d,
        "clave_publica": (e, n),
        "clave_privada": (d, n),
    }


claves = generar_claves_rsa(p=5, q=11, e=3)
for k, v in claves.items():
    print(f"{k:15s} = {v}")

p               = 5
q               = 11
n               = 55
phi             = 40
e               = 3
d               = 27
clave_publica   = (3, 55)
clave_privada   = (27, 55)


### 1.1 Cifrado y descifrado

Con las claves generadas, ciframos un mensaje $m$ (un número entero menor que $n$):

$$c = m^e \bmod n \qquad\qquad m = c^d \bmod n$$

In [2]:
def cifrar_rsa(m, clave_publica):
    e, n = clave_publica
    if m >= n:
        raise ValueError(f"El mensaje m={m} debe ser menor que n={n}")
    return pow(m, e, n)


def descifrar_rsa(c, clave_privada):
    d, n = clave_privada
    return pow(c, d, n)


mensaje = 10
c = cifrar_rsa(mensaje, claves["clave_publica"])
m_recuperado = descifrar_rsa(c, claves["clave_privada"])

print(f"Mensaje original   : {mensaje}")
print(f"Mensaje cifrado    : {c}")
print(f"Mensaje recuperado : {m_recuperado}")
assert m_recuperado == mensaje, "¡Algo ha ido mal! El mensaje recuperado no coincide."
print("\n✅ El mensaje se ha cifrado y descifrado correctamente.")

Mensaje original   : 10
Mensaje cifrado    : 10
Mensaje recuperado : 10

✅ El mensaje se ha cifrado y descifrado correctamente.


### 1.2 Probar con varios mensajes

Vamos a comprobar que el cifrado y descifrado funciona para distintos mensajes $m < n$.

In [3]:
for mensaje in [0, 1, 2, 7, 10, 25, 54]:
    c = cifrar_rsa(mensaje, claves["clave_publica"])
    m_recuperado = descifrar_rsa(c, claves["clave_privada"])
    estado = "OK" if m_recuperado == mensaje else "FALLO"
    print(f"m={mensaje:>3} -> cifrado={c:>3} -> descifrado={m_recuperado:>3}  [{estado}]")

m=  0 -> cifrado=  0 -> descifrado=  0  [OK]
m=  1 -> cifrado=  1 -> descifrado=  1  [OK]
m=  2 -> cifrado=  8 -> descifrado=  2  [OK]
m=  7 -> cifrado= 13 -> descifrado=  7  [OK]
m= 10 -> cifrado= 10 -> descifrado= 10  [OK]
m= 25 -> cifrado=  5 -> descifrado= 25  [OK]
m= 54 -> cifrado= 54 -> descifrado= 54  [OK]


### 1.3 Firma digital simplificada

En RSA, firmar es "cifrar con la clave privada" y verificar es "descifrar con la clave pública" (en la práctica real se firma un *hash* del mensaje, no el mensaje completo, y se usa el esquema RSA-PSS; aquí lo simplificamos con fines didácticos).

$$s = h^d \bmod n \qquad\qquad h' = s^e \bmod n$$

Si $h' = h$, la firma es válida.

In [4]:
import hashlib

def hash_mensaje(mensaje_texto, n):
    """Convierte un mensaje de texto en un número entero menor que n usando SHA-256."""
    digest = hashlib.sha256(mensaje_texto.encode("utf-8")).hexdigest()
    return int(digest, 16) % n


def firmar_rsa(mensaje_texto, clave_privada, n):
    d, _ = clave_privada
    h = hash_mensaje(mensaje_texto, n)
    firma = pow(h, d, n)
    return firma, h


def verificar_firma_rsa(mensaje_texto, firma, clave_publica, n):
    e, _ = clave_publica
    h_esperado = hash_mensaje(mensaje_texto, n)
    h_obtenido = pow(firma, e, n)
    return h_obtenido == h_esperado


documento = "Contrato firmado por Alice el 12/09/2026"
firma, h_original = firmar_rsa(documento, claves["clave_privada"], claves["n"])
es_valida = verificar_firma_rsa(documento, firma, claves["clave_publica"], claves["n"])

print(f"Documento          : {documento!r}")
print(f"Hash reducido mod n: {h_original}")
print(f"Firma generada     : {firma}")
print(f"¿Firma válida?      : {es_valida}")

# Si alguien modifica el documento, la verificación debe fallar
documento_alterado = documento + " (modificado por un atacante)"
print("\n--- Intento de verificar con el documento alterado ---")
print(f"¿Firma válida sobre el documento alterado? {verificar_firma_rsa(documento_alterado, firma, claves['clave_publica'], claves['n'])}")

Documento          : 'Contrato firmado por Alice el 12/09/2026'
Hash reducido mod n: 18
Firma generada     : 17
¿Firma válida?      : True

--- Intento de verificar con el documento alterado ---
¿Firma válida sobre el documento alterado? False


> **Nota de seguridad:** este ejemplo usa números muy pequeños y una reducción simplificada del hash, solo para fines educativos. En un sistema real, RSA se implementa con primos de cientos de dígitos y esquemas de relleno seguro como **RSA-OAEP** (cifrado) y **RSA-PSS** (firma), disponibles en librerías como `cryptography` en Python.

---
## Parte 2. LFSR (Linear Feedback Shift Register)

Un **LFSR** es un registro de $n$ bits que en cada paso:

1. calcula un nuevo bit de entrada aplicando XOR sobre un subconjunto fijo de posiciones del registro (llamado **polinomio de realimentación** o *taps*);
2. desplaza todos los bits una posición;
3. inserta el nuevo bit calculado.

Históricamente se ha usado para generar secuencias pseudoaleatorias en cifrados de flujo (por ejemplo, en el diseño original de A5/1 en telefonía móvil GSM). Es muy eficiente en hardware, pero **matemáticamente predecible**: si un atacante observa suficientes bits de salida, puede reconstruir el estado interno completo del LFSR.

In [5]:
def lfsr_paso(estado, taps):
    """Ejecuta un paso de un LFSR tipo Fibonacci.

    estado: lista de bits (0/1), el más a la izquierda es el bit más significativo.
    taps: posiciones (índices, empezando en 0) del estado que participan en el XOR de realimentación.
    Devuelve (nuevo_estado, bit_de_salida).
    """
    bit_salida = estado[-1]  # el bit que "sale" del registro
    nuevo_bit = 0
    for t in taps:
        nuevo_bit ^= estado[t]
    nuevo_estado = [nuevo_bit] + estado[:-1]
    return nuevo_estado, bit_salida


def generar_secuencia_lfsr(semilla, taps, num_bits):
    estado = list(semilla)
    salida = []
    for _ in range(num_bits):
        estado, bit = lfsr_paso(estado, taps)
        salida.append(bit)
    return salida


# LFSR de 4 bits con polinomio de realimentación x^4 + x^3 + 1 (taps en posiciones 0 y 1)
semilla = [1, 0, 0, 0]  # estado inicial, no puede ser todo ceros
taps = [0, 1]

secuencia = generar_secuencia_lfsr(semilla, taps, num_bits=20)
print("Semilla inicial :", semilla)
print("Taps (realim.)  :", taps)
print("Secuencia de bits generada:")
print("".join(str(b) for b in secuencia))

Semilla inicial : [1, 0, 0, 0]
Taps (realim.)  : [0, 1]
Secuencia de bits generada:
00011011011011011011


### 2.1 Periodo del LFSR

Un LFSR de $n$ bits tiene, como máximo, $2^n - 1$ estados posibles antes de repetirse (se excluye el estado con todo ceros, que quedaría "atascado"). Vamos a comprobarlo generando bits hasta que el estado se repita.

In [6]:
def periodo_lfsr(semilla, taps, max_pasos=1000):
    estado = list(semilla)
    vistos = {tuple(estado): 0}
    for paso in range(1, max_pasos + 1):
        estado, _ = lfsr_paso(estado, taps)
        clave = tuple(estado)
        if clave in vistos:
            return paso - vistos[clave]
        vistos[clave] = paso
    return None  # no se repitió dentro de max_pasos


n_bits = len(semilla)
periodo = periodo_lfsr(semilla, taps)
periodo_maximo_teorico = 2 ** n_bits - 1

print(f"Registro de {n_bits} bits")
print(f"Periodo observado        : {periodo}")
print(f"Periodo máximo teórico   : 2^{n_bits} - 1 = {periodo_maximo_teorico}")
if periodo == periodo_maximo_teorico:
    print("✅ Este LFSR es de 'periodo máximo': recorre todos los estados posibles antes de repetirse.")
else:
    print("⚠️ Este LFSR no alcanza el periodo máximo con estos taps.")

Registro de 4 bits
Periodo observado        : 3
Periodo máximo teórico   : 2^4 - 1 = 15
⚠️ Este LFSR no alcanza el periodo máximo con estos taps.


### 2.2 ¿Por qué un LFSR no es seguro por sí solo?

Aunque la secuencia generada "parece" aleatoria a simple vista, un LFSR es **completamente lineal**: cada nuevo bit es una combinación XOR de bits anteriores. Esto significa que, si un atacante conoce (u observa) $2n$ bits consecutivos de la salida de un LFSR de $n$ bits, puede reconstruir exactamente los `taps` y el estado interno completo, y predecir el resto de la secuencia.

Vamos a demostrarlo con un pequeño ataque usando álgebra lineal sobre $\mathbb{F}_2$ (aritmética módulo 2), sin necesitar librerías externas.

In [7]:
def resolver_sistema_gf2(filas, terminos_independientes):
    """Resuelve un sistema lineal sobre GF(2) mediante eliminación gaussiana simple.

    filas: lista de listas de 0/1 (cada fila son los coeficientes de una ecuación).
    terminos_independientes: lista de 0/1 (el resultado de cada ecuación).
    Devuelve la solución como lista de 0/1, o None si el sistema no tiene solución única.
    """
    n = len(filas[0])
    matriz = [fila[:] + [b] for fila, b in zip(filas, terminos_independientes)]

    fila_actual = 0
    for col in range(n):
        # buscar una fila con un 1 en esta columna, a partir de fila_actual
        pivote = None
        for r in range(fila_actual, len(matriz)):
            if matriz[r][col] == 1:
                pivote = r
                break
        if pivote is None:
            continue
        matriz[fila_actual], matriz[pivote] = matriz[pivote], matriz[fila_actual]
        for r in range(len(matriz)):
            if r != fila_actual and matriz[r][col] == 1:
                matriz[r] = [(a ^ b) for a, b in zip(matriz[r], matriz[fila_actual])]
        fila_actual += 1

    solucion = [fila[-1] for fila in matriz[:n]]
    return solucion


def romper_lfsr(bits_observados, n_bits):
    """Dado un flujo de al menos 2*n_bits observados, recupera los taps del LFSR
    asumiendo realimentación tipo Fibonacci sobre un registro de tamaño n_bits.
    """
    if len(bits_observados) < 2 * n_bits:
        raise ValueError("Se necesitan al menos 2*n_bits bits observados para romper el LFSR")

    # Cada ventana de n_bits consecutivos permite plantear una ecuación lineal:
    # el bit siguiente es una combinación XOR de esos n_bits.
    filas = [bits_observados[i:i + n_bits] for i in range(n_bits)]
    resultados = [bits_observados[i + n_bits] for i in range(n_bits)]

    coeficientes = resolver_sistema_gf2(filas, resultados)

    # Los coeficientes recuperados indexan la ventana en el mismo orden en que se
    # observan los bits (de más antiguo a más reciente). El LFSR original define
    # los taps sobre las posiciones del estado interno (de MSB a LSB), que es el
    # orden inverso. Convertimos para poder comparar ambos conjuntos directamente.
    return sorted(n_bits - 1 - j for j, bit in enumerate(coeficientes) if bit == 1)


# Generamos una secuencia "espiada" (observada por un atacante) con el LFSR original
bits_observados = generar_secuencia_lfsr(semilla, taps, num_bits=2 * n_bits + 4)

taps_recuperados = romper_lfsr(bits_observados, n_bits)

print("Bits observados por el atacante:", "".join(str(b) for b in bits_observados))
print(f"Taps originales     : {sorted(taps)}")
print(f"Taps recuperados    : {taps_recuperados}")
print("✅ ¡El atacante ha reconstruido los taps solo observando la salida!" if sorted(taps) == taps_recuperados else "⚠️ No coinciden, revisa el ejemplo.")


Bits observados por el atacante: 000110110110
Taps originales     : [0, 1]
Taps recuperados    : [0, 1]
✅ ¡El atacante ha reconstruido los taps solo observando la salida!


### 2.3 Conclusión de la parte 2

Este pequeño "ataque" (una eliminación gaussiana sobre GF(2)) demuestra por qué los LFSR **no se usan solos** como generadores criptográficos: su linealidad los hace completamente predecibles a partir de muy pocos bits observados.

En la práctica, los cifrados de flujo modernos combinan varios LFSR con funciones **no lineales** (por ejemplo, A5/1 combina tres LFSR con una función de mayoría), o directamente se sustituyen por generadores criptográficamente seguros como **ChaCha20**, que no tienen esta debilidad.

---
## Resumen final

| Mecanismo | Tipo | Basado en | Seguro por sí solo |
|---|---|---|---|
| RSA | Asimétrico | Factorización de enteros | Sí, con relleno seguro (OAEP/PSS) |
| LFSR | Generador de flujo | Álgebra lineal (XOR) | No, es predecible con pocos bits |

**Idea clave:** que una secuencia "parezca" aleatoria no significa que sea segura. La seguridad criptográfica exige resistencia frente a un atacante que analiza la estructura matemática del sistema, no solo frente a la inspección visual.